<a href="https://colab.research.google.com/github/Fatou-Kine3/github_task/blob/main/credit_information_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Problem 1 – Competition Overview

The goal of the Home Credit Default Risk competition is to predict the
probability that a loan applicant will have difficulty repaying a loan.

`TARGET = 1` represents a client who experienced repayment difficulties,
while `TARGET = 0` represents a client without repayment difficulties.

The submission must contain a predicted probability for each customer,
rather than a binary class label (0 or 1).

The submissions are evaluated using the AUC (Area Under the ROC Curve).
A higher AUC indicates better discrimination between the two classes.

In [1]:
import os

os.listdir()

['.config', 'application_train.csv.zip', 'sample_data']

In [2]:
import os

os.listdir()

['.config',
 'sample_submission.csv',
 'application_train.csv.zip',
 'application_test.csv.zip',
 'sample_data']

In [3]:
import zipfile

with zipfile.ZipFile("application_train.csv.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [4]:
import os

os.listdir()

['.config',
 'application_train.csv',
 'sample_submission.csv',
 'application_train.csv.zip',
 'application_test.csv.zip',
 'sample_data']

In [5]:
import pandas as pd

df = pd.read_csv("application_train.csv")

print(df.shape)
df.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
print("Nombre de lignes et colonnes :", df.shape)

print("\nDistribution de TARGET :")
print(df["TARGET"].value_counts())

print("\nPourcentage de TARGET :")
print(df["TARGET"].value_counts(normalize=True))

Nombre de lignes et colonnes : (307511, 122)

Distribution de TARGET :
TARGET
0    282686
1     24825
Name: count, dtype: int64

Pourcentage de TARGET :
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [7]:
X = df.drop("TARGET", axis=1)
y = df["TARGET"]

print("X :", X.shape)
print("y :", y.shape)

X : (307511, 121)
y : (307511,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training :", X_train.shape)
print("Validation :", X_val.shape)

Training : (230633, 121)
Validation : (76878, 121)


In [9]:
print("Training:")
print(y_train.value_counts(normalize=True))

print("\nValidation:")
print(y_val.value_counts(normalize=True))

Training:
TARGET
0    0.91927
1    0.08073
Name: proportion, dtype: float64

Validation:
TARGET
0    0.919275
1    0.080725
Name: proportion, dtype: float64


In [10]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

print("Variables numériques :", len(numeric_features))
print("Variables catégorielles :", len(categorical_features))

Variables numériques : 105
Variables catégorielles : 16


In [11]:
print("Quelques variables numériques :")
print(numeric_features[:10])

print("\nQuelques variables catégorielles :")
print(categorical_features[:10])

Quelques variables numériques :
Index(['SK_ID_CURR', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
       'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
       'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION'],
      dtype='object')

Quelques variables catégorielles :
Index(['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
       'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'],
      dtype='object')


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

print("Training après prétraitement :", X_train_processed.shape)
print("Validation après prétraitement :", X_val_processed.shape)

Training après prétraitement : (230633, 245)
Validation après prétraitement : (76878, 245)


In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [15]:
model.fit(X_train_processed, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [16]:
y_val_proba = model.predict_proba(X_val_processed)[:, 1]

print(y_val_proba[:10])

[0.23946177 0.04817657 0.08488503 0.05582614 0.18975691 0.0230584
 0.00867834 0.01449453 0.04909577 0.02141384]


In [17]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_val, y_val_proba)

print("Validation AUC:", auc)

Validation AUC: 0.7483284496274647


In [ ]:
### Baseline Model Result

Logistic Regression was used as the baseline model.

The model was trained on the training data and evaluated on the
validation data.

The model was evaluated using ROC-AUC, based on the predicted
probability of `TARGET = 1`.

The validation AUC was:

**AUC = [enter your result here]**

This value serves as the baseline for comparing the models in the
following experiments.

In [18]:
import zipfile

with zipfile.ZipFile("application_test.csv.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [19]:
import os

print("application_test.csv" in os.listdir())

True


In [20]:
import pandas as pd

test_df = pd.read_csv("application_test.csv")

print(test_df.shape)
test_df.head()

(48744, 121)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
X_test_processed = preprocessor.transform(test_df)

print(X_test_processed.shape)

(48744, 245)


In [22]:
test_proba = model.predict_proba(X_test_processed)[:, 1]

print(test_proba[:10])

[0.05929716 0.25295312 0.04638312 0.03287053 0.12757186 0.03491687
 0.02164185 0.09997615 0.01545005 0.10751683]


In [23]:
sample_submission = pd.read_csv("sample_submission.csv")

print(sample_submission.shape)
print(sample_submission.head())
print(sample_submission.columns)

(48744, 2)
   SK_ID_CURR  TARGET
0      100001     0.5
1      100005     0.5
2      100013     0.5
3      100028     0.5
4      100038     0.5
Index(['SK_ID_CURR', 'TARGET'], dtype='object')


In [24]:
submission = sample_submission.copy()

submission["TARGET"] = test_proba

print(submission.head())

   SK_ID_CURR    TARGET
0      100001  0.059297
1      100005  0.252953
2      100013  0.046383
3      100028  0.032871
4      100038  0.127572


In [25]:
print("Nombre de lignes :", len(submission))
print("Nombre de lignes test :", len(test_df))

print("\nColonnes :")
print(submission.columns)

print("\nValeurs manquantes :")
print(submission.isnull().sum())

Nombre de lignes : 48744
Nombre de lignes test : 48744

Colonnes :
Index(['SK_ID_CURR', 'TARGET'], dtype='object')

Valeurs manquantes :
SK_ID_CURR    0
TARGET        0
dtype: int64


In [26]:
print(
    (submission["SK_ID_CURR"] == test_df["SK_ID_CURR"]).all()
)

True


In [27]:
submission.to_csv(
    "submission.csv",
    index=False
)

In [28]:
final_submission = pd.read_csv("submission.csv")

print(final_submission.shape)
print(final_submission.head())

(48744, 2)
   SK_ID_CURR    TARGET
0      100001  0.059297
1      100005  0.252953
2      100013  0.046383
3      100028  0.032871
4      100038  0.127572


In [29]:
from google.colab import files

files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
random_state=42
stratify=y

In [31]:
features_exp1 = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

X_exp1 = df[features_exp1]
y = df["TARGET"]

print(X_exp1.head())
print(X_exp1.shape)

   AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  DAYS_BIRTH  DAYS_EMPLOYED
0          202500.0    406597.5      24700.5       -9461           -637
1          270000.0   1293502.5      35698.5      -16765          -1188
2           67500.0    135000.0       6750.0      -19046           -225
3          135000.0    312682.5      29686.5      -19005          -3039
4          121500.0    513000.0      21865.5      -19932          -3038
(307511, 5)


In [32]:
X_train_exp1, X_val_exp1, y_train_exp1, y_val_exp1 = train_test_split(
    X_exp1,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [33]:
preprocessor_exp1 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [34]:
X_train_exp1_processed = preprocessor_exp1.fit_transform(X_train_exp1)
X_val_exp1_processed = preprocessor_exp1.transform(X_val_exp1)

In [35]:
model_exp1 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_exp1.fit(X_train_exp1_processed, y_train_exp1)

LogisticRegression(max_iter=1000, random_state=42)

In [36]:
y_pred_exp1 = model_exp1.predict_proba(
    X_val_exp1_processed
)[:, 1]

auc_exp1 = roc_auc_score(
    y_val_exp1,
    y_pred_exp1
)

print("Experiment 1 AUC:", auc_exp1)

Experiment 1 AUC: 0.5859024393856362


In [37]:
results = pd.DataFrame({
    "Experiment": ["Experiment 1"],
    "Features": [", ".join(features_exp1)],
    "Random State": [42],
    "AUC": [auc_exp1]
})

results

,Experiment,Features,Random State,AUC
0,Experiment 1,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.585902


In [38]:
features_exp2 = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "REGION_POPULATION_RELATIVE"
]

X_exp2 = df[features_exp2]
y = df["TARGET"]

print(X_exp2.shape)

(307511, 8)


In [39]:
X_train_exp2, X_val_exp2, y_train_exp2, y_val_exp2 = train_test_split(
    X_exp2,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [40]:
preprocessor_exp2 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_exp2_processed = preprocessor_exp2.fit_transform(X_train_exp2)
X_val_exp2_processed = preprocessor_exp2.transform(X_val_exp2)

In [41]:
model_exp2 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_exp2.fit(X_train_exp2_processed, y_train_exp2)

LogisticRegression(max_iter=1000, random_state=42)

In [42]:
y_pred_exp2 = model_exp2.predict_proba(
    X_val_exp2_processed
)[:, 1]

auc_exp2 = roc_auc_score(
    y_val_exp2,
    y_pred_exp2
)

print("Experiment 2 AUC:", auc_exp2)

Experiment 2 AUC: 0.5933506638831556


In [43]:
results.loc[len(results)] = [
    "Experiment 2",
    ", ".join(features_exp2),
    42,
    auc_exp2
]

results

,Experiment,Features,Random State,AUC
0,Experiment 1,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.585902
1,Experiment 2,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.593351


In [44]:
features_exp3 = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "REGION_POPULATION_RELATIVE",
    "OWN_CAR_AGE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3"
]

X_exp3 = df[features_exp3]
y = df["TARGET"]

print(X_exp3.shape)

(307511, 12)


In [45]:
X_train_exp3, X_val_exp3, y_train_exp3, y_val_exp3 = train_test_split(
    X_exp3,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [46]:
preprocessor_exp3 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_exp3_processed = preprocessor_exp3.fit_transform(X_train_exp3)
X_val_exp3_processed = preprocessor_exp3.transform(X_val_exp3)

In [47]:
model_exp3 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_exp3.fit(X_train_exp3_processed, y_train_exp3)

LogisticRegression(max_iter=1000, random_state=42)

In [48]:
y_pred_exp3 = model_exp3.predict_proba(
    X_val_exp3_processed
)[:, 1]

auc_exp3 = roc_auc_score(
    y_val_exp3,
    y_pred_exp3
)

print("Experiment 3 AUC:", auc_exp3)

Experiment 3 AUC: 0.7223108163928209


In [49]:
results.loc[len(results)] = [
    "Experiment 3",
    ", ".join(features_exp3),
    42,
    auc_exp3
]

results

,Experiment,Features,Random State,AUC
0,Experiment 1,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.585902
1,Experiment 2,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.593351
2,Experiment 3,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.722311


In [50]:
features_exp4 = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "REGION_POPULATION_RELATIVE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE"
]

X_exp4 = df[features_exp4]
y = df["TARGET"]

print(X_exp4.shape)

(307511, 15)


In [51]:
X_train_exp4, X_val_exp4, y_train_exp4, y_val_exp4 = train_test_split(
    X_exp4,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [52]:
preprocessor_exp4 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

X_train_exp4_processed = preprocessor_exp4.fit_transform(X_train_exp4)
X_val_exp4_processed = preprocessor_exp4.transform(X_val_exp4)

In [53]:
model_exp4 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_exp4.fit(
    X_train_exp4_processed,
    y_train_exp4
)

LogisticRegression(max_iter=1000, random_state=42)

In [54]:
y_pred_exp4 = model_exp4.predict_proba(
    X_val_exp4_processed
)[:, 1]

auc_exp4 = roc_auc_score(
    y_val_exp4,
    y_pred_exp4
)

print("Experiment 4 AUC:", auc_exp4)

Experiment 4 AUC: 0.7296791873471603


In [55]:
results.loc[len(results)] = [
    "Experiment 4",
    ", ".join(features_exp4),
    42,
    auc_exp4
]

results

,Experiment,Features,Random State,AUC
0,Experiment 1,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.585902
1,Experiment 2,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.593351
2,Experiment 3,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.722311
3,Experiment 4,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT...",42,0.729679


In [56]:
features_exp5 = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "REGION_POPULATION_RELATIVE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_FAMILY_STATUS",
    "NAME_EDUCATION_TYPE"
]

X_exp5 = df[features_exp5]
y = df["TARGET"]

print(X_exp5.shape)

(307511, 19)


In [57]:
X_train_exp5, X_val_exp5, y_train_exp5, y_val_exp5 = train_test_split(
    X_exp5,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [58]:
numeric_features_exp5 = X_exp5.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_exp5 = X_exp5.select_dtypes(
    include=["object"]
).columns.tolist()

print("Variables numériques :", numeric_features_exp5)
print("Variables catégorielles :", categorical_features_exp5)

Variables numériques : ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'REGION_POPULATION_RELATIVE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']
Variables catégorielles : ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_EDUCATION_TYPE']


In [59]:
numeric_pipeline_exp5 = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_exp5 = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [60]:
preprocessor_exp5 = ColumnTransformer([
    ("num", numeric_pipeline_exp5, numeric_features_exp5),
    ("cat", categorical_pipeline_exp5, categorical_features_exp5)
])

In [61]:
X_train_exp5_processed = preprocessor_exp5.fit_transform(
    X_train_exp5
)

X_val_exp5_processed = preprocessor_exp5.transform(
    X_val_exp5
)

In [62]:
model_exp5 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_exp5.fit(
    X_train_exp5_processed,
    y_train_exp5
)

LogisticRegression(max_iter=1000, random_state=42)

In [63]:
y_pred_exp5 = model_exp5.predict_proba(
    X_val_exp5_processed
)[:, 1]

auc_exp5 = roc_auc_score(
    y_val_exp5,
    y_pred_exp5
)

print("Experiment 5 AUC:", auc_exp5)

Experiment 5 AUC: 0.7388723587111905


In [64]:
results.loc[len(results)] = [
    "Experiment 5",
    ", ".join(features_exp5),
    42,
    auc_exp5
]

results

,Experiment,Features,Random State,AUC
0,Experiment 1,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.585902
1,Experiment 2,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.593351
2,Experiment 3,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, DAY...",42,0.722311
3,Experiment 4,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT...",42,0.729679
4,Experiment 5,"AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY, AMT...",42,0.738872


In [ ]:
### Conclusion

Five training and validation experiments were performed using the same
`random_state=42` and the same training/validation split.

The AUC increased from 0.585902 in Experiment 1 to 0.738872 in
Experiment 5.

Experiment 5 achieved the best performance, with a validation AUC of
0.738872. This improvement is likely related to the use of additional
features and the encoding of categorical variables.

The numerical variables were processed using median imputation and
standardization. The categorical variables were processed using
most-frequent imputation and One-Hot Encoding.

The preprocessing was fitted only on the training data and then applied
to the validation data in order to avoid data leakage.

However, a higher validation AUC does not necessarily guarantee better
performance on unseen data. Therefore, the results should be interpreted
with caution.